# Mr. Chicken - Lip-Sync com Wav2Lip (Google Colab)

Este notebook foi gerado pelo **Mr. Chicken** para permitir a sincronização labial gratuita usando o processamento em GPU do Google Colab.

### **Instruções de Uso:**
1. Vá no menu superior e clique em **Ambiente de execução** -> **Alterar tipo de ambiente de execução** e certifique-se de que a **GPU** (T4 GPU) está selecionada.
2. Execute as células abaixo em ordem clicando no botão **Play [ ]** à esquerda de cada célula.
3. Faça o upload do **Avatar** e do **Áudio de Voz** quando solicitado.
4. Aguarde o processamento terminar e faça o download do arquivo `result.mp4` gerado.
5. Envie o arquivo final de volta no painel do Mr. Chicken para concluir a renderização!

### **Passo 1: Instalar e configurar o Wav2Lip**

In [ ]:
import os
import shutil

# 1. Resetar diretório de trabalho no Colab para evitar clonagem aninhada recursiva
if os.path.exists('/content'):
    %cd /content

# 2. Clonar o repositório apenas se não existir
if not os.path.exists('Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

%cd Wav2Lip

# 3. Criar diretório para checkpoints
checkpoint_path = "checkpoints/wav2lip_gan.pth"
os.makedirs("checkpoints", exist_ok=True)

# Lista de comandos de download com mirrors de fallback (Hugging Face via curl + Google Drive via gdown)
download_commands = [
    # Mirror 1: Hugging Face snehilsanyal usando curl (bypassa 401 com User-Agent)
    'curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" "https://huggingface.co/snehilsanyal/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -o checkpoints/wav2lip_gan.pth',
    
    # Mirror 2: Hugging Face briaai usando curl
    'curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" "https://huggingface.co/briaai/Wav2Lip/resolve/main/wav2lip_gan.pth" -o checkpoints/wav2lip_gan.pth',
    
    # Mirror 3: Google Drive 1 (gdown)
    'gdown --id 1dwHujX7RVNCvdR1RR93z0FS2T2yzqup9 --output checkpoints/wav2lip_gan.pth',
    
    # Mirror 4: Google Drive 2 (gdown)
    'gdown --id 10Iu05Modfti3pDbxCFPnofmfVlbkvrCm --output checkpoints/wav2lip_gan.pth'
]

download_success = False
for cmd in download_commands:
    print(f"Executando download via mirror: {cmd[:60]}...")
    try:
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
            
        # Executa o comando de terminal de download
        os.system(cmd)
        
        # Verifica se o arquivo baixado existe e tem tamanho esperado (> 100MB)
        if os.path.exists(checkpoint_path):
            size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
            if size_mb > 100:
                print(f"✅ Download do checkpoint concluído com sucesso ({size_mb:.1f} MB)!")
                download_success = True
                break
            else:
                print(f"⚠️ Arquivo muito pequeno ({size_mb:.4f} MB). Provavelmente LFS pointer ou redirecionamento falho. Tentando próximo mirror...")
        else:
            print("⚠️ Arquivo não foi criado. Tentando próximo mirror...")
    except Exception as e:
        print(f"❌ Erro ao baixar deste mirror: {e}")

if not download_success:
    raise RuntimeError("Não foi possível baixar o checkpoint de nenhum dos mirrors públicos. Verifique se o Hugging Face ou Google Drive estão acessíveis.")

# 4. Corrigir incompatibilidades de bibliotecas antigas com o NumPy moderno no arquivo principal
with open("inference.py", "r") as f:
    inf_content = f.read()

# Injeta o patch do NumPy no topo do inference.py
numpy_patch = "import numpy as np\nnp.complex = complex\nnp.float = float\nnp.int = int\n"
if "np.complex = complex" not in inf_content:
    inf_content = numpy_patch + inf_content

# Corrige a definição do argumento --static que foi declarado incorretamente como type=bool
inf_content = inf_content.replace("type=bool, default=False", "action='store_true'")
inf_content = inf_content.replace("type=bool,default=False", "action='store_true'")
inf_content = inf_content.replace("type=bool", "action='store_true'")

with open("inference.py", "w") as f:
    f.write(inf_content)

# Corrige np.int por int no arquivo audio.py
with open("audio.py", "r") as f:
    audio_content = f.read()

audio_content = audio_content.replace("np.int", "int")

with open("audio.py", "w") as f:
    f.write(audio_content)

print("✅ Repositório clonado, corrigido e adaptado com sucesso!")

### **Passo 2: Baixar requerimentos adicionais**

In [ ]:
# Instalar requisitos do Wav2Lip
!pip install -q -r requirements.txt
# Instalar librosa antigo que é compatível com Wav2Lip ou ajustar
!pip install -q librosa==0.8.0
print("✅ Dependências instaladas com sucesso!")

### **Passo 3: Fazer Upload do Avatar e do Áudio do Mr. Chicken**
Execute a célula abaixo. Ela abrirá caixas de seleção de arquivos. Selecione:
1. O arquivo do avatar (imagem `.jpg`/`.png` ou vídeo `.mp4`).
2. O arquivo de áudio de voz gerado (`.wav` ou `.mp3`).

In [ ]:
from google.colab import files
import os
import shutil

# Limpar uploads anteriores se houver
for f in ['input_avatar', 'input_audio']:
    for ext in ['.png', '.jpg', '.jpeg', '.webp', '.mp4', '.avi', '.mov', '.wav', '.mp3']:
        if os.path.exists(f + ext):
            os.remove(f + ext)

print("1. FAÇA UPLOAD DO AVATAR (Vídeo ou Imagem):")
up_avatar = files.upload()
avatar_name = list(up_avatar.keys())[0]
avatar_ext = os.path.splitext(avatar_name)[1].lower()
shutil.move(avatar_name, "input_avatar" + avatar_ext)

print("\n2. FAÇA UPLOAD DO ÁUDIO DE VOZ:")
up_audio = files.upload()
audio_name = list(up_audio.keys())[0]
audio_ext = os.path.splitext(audio_name)[1].lower()
shutil.move(audio_name, "input_audio" + audio_ext)

# Converter áudio para .wav para garantir 100% de compatibilidade
!ffmpeg -y -i "input_audio{audio_ext}" -ac 1 -ar 16000 input_audio_processed.wav

print("\n✅ Arquivos carregados e preparados!")

### **Passo 4: Executar a Sincronização Labial (Lip-Sync)**
Execute esta célula para rodar o modelo Wav2Lip.

In [ ]:
import glob

# Identificar arquivo de avatar
avatar_files = glob.glob("input_avatar.*")[0]
avatar_ext = os.path.splitext(avatar_files)[1].lower()
is_image = avatar_ext in ['.png', '.jpg', '.jpeg', '.webp']

cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {avatar_files} --audio input_audio_processed.wav --outfile result.mp4"
if is_image:
    cmd += " --static"

print(f"Executando comando: {cmd}")
!{cmd}

if os.path.exists("result.mp4"):
    print("\n✅ Lip-sync concluído com sucesso! result.mp4 gerado.")
else:
    print("\n❌ Falha ao gerar o vídeo. Verifique se o rosto no avatar está bem visível e frontal.")

### **Passo 5: Fazer o Download do Vídeo Final**

In [ ]:
if os.path.exists("result.mp4"):
    print("Iniciando download do vídeo...")
    files.download("result.mp4")
else:
    print("Arquivo result.mp4 não encontrado.")